In [1]:
import pandas as pd
import os
import csv
import re
import ftfy
import unicodedata
from collections import Counter
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.trainers import BpeTrainer
from tokenizers.processors import TemplateProcessing
from tokenizers import Tokenizer
import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
from sklearn.model_selection import train_test_split
import time

torch.manual_seed(42)

BATCH_SIZE     = 32
CONTEXT_SIZE   = 256
MAX_ITERS      = 50000
EVAL_INTERVAL  = 500
LR_RATE        = 3e-4
EVAL_ITERS     = 10
device = 'cuda' if torch.cuda.is_available() else 'cpu'



# Training Code

In [2]:
tokenizer = Tokenizer.from_file(
    r'..\data\processed\tokenizer.json'
)
tokenizer.no_padding()

In [3]:
PAD_TOKEN_ID = tokenizer.token_to_id("[PAD]")
PAD_TOKEN_ID

2

In [4]:
def get_tokenized_data():
    cache_path = r'..\data\processed\tokenized_jokes_split_nopad.npz'

    if os.path.exists(cache_path):
        print("Loading tokenized data from cache...")
        data = np.load(cache_path)
        train_data = torch.from_numpy(data['train_data'])
        test_data  = torch.from_numpy(data['test_data'])
    else:
        data = pd.read_csv(r'..\data\processed\final_cleaned_jokes.csv')
        data = data[data.joke_cleaned.notnull()]
        tokenizer = Tokenizer.from_file(
            r'..\data\processed\tokenizer.json'
        )
        tokenizer.no_padding()
        sequences = [f"tell me a joke about {t} [JOKE] {j}" 
                 for t,j in zip(data.topic, data.joke_cleaned)]
        tokenized = tokenizer.encode_batch(sequences)
        tokenized_ids = [x.ids for x in tokenized]
        tokenized_ids = [x for xs in tokenized_ids for x in xs]
        tokenized_ids = torch.tensor(tokenized_ids)
        
        data_len = len(tokenized_ids)
        train_data = tokenized_ids[:int(data_len*.9)]
        test_data  = tokenized_ids[int(data_len*.9):]
        np.savez_compressed(
            cache_path,
            train_data=train_data.numpy(),
            test_data=test_data.numpy()
        )
    return train_data, test_data

train_data, test_data = get_tokenized_data()

Loading tokenized data from cache...


In [5]:
def get_batch(context_size=CONTEXT_SIZE, batch_size=BATCH_SIZE, data='train'):
    dataset = train_data if data == 'train' else test_data
    idx = torch.randint(len(dataset) - context_size - 1, (batch_size,))

    # build batch using stacking
    x = torch.stack([dataset[i : i + context_size] for i in idx])
    y = torch.stack([dataset[i + 1 : i + 1 + context_size] for i in idx])

    x = x.to(device)
    y = y.to(device)
    return x, y
xb, yb = get_batch()
print(xb)
print(yb)

tensor([[  271,   219,  1517,  ...,    73,    74,   185],
        [  207,  1460,   388,  ...,    58,   806,    61],
        [  249, 16591,  5495,  ...,  3257,    95,  7608],
        ...,
        [  373,   140,    56,  ...,   249,   100,  7731],
        [  373,    89,    62,  ...,    62,     1,     0],
        [   95,  5107,   179,  ...,  2732,  1209,    80]], device='cuda:0')
tensor([[  219,  1517,    89,  ...,    74,   185,   205],
        [ 1460,   388,   200,  ...,   806,    61,  7727],
        [16591,  5495,  1121,  ...,    95,  7608,    62],
        ...,
        [  140,    56,   403,  ...,   100,  7731,    12],
        [   89,    62,    89,  ...,     1,     0,   373],
        [ 5107,   179, 19000,  ...,  1209,    80,   140]], device='cuda:0')


In [6]:
@torch.no_grad()
def estimate_loss(context_size=CONTEXT_SIZE):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            X, Y = get_batch(context_size=context_size,data=split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


class Head(nn.Module):
    def __init__(self, emb_dim, context_size, head_size, dropout):
        super().__init__()
        self.key   = nn.Linear(emb_dim, head_size, bias=False)
        self.query = nn.Linear(emb_dim, head_size, bias=False)
        self.value = nn.Linear(emb_dim, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(context_size, context_size)))

        self.dropout = nn.Dropout(dropout)

        
    def forward(self, x, attn_mask=None):
        B, T, E = x.shape
        k = self.key(x)    # (B, T, H)
        q = self.query(x)  # (B, T, H)

        weights = q @ k.transpose(-2, -1) * E**-0.5 # (B, T, H) @ (B, H, T) -> (B, T, T)
        weights = weights.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        
        if attn_mask is not None:
            pad_mask = attn_mask[:, None, :].to(weights.device)  # (B, 1, T)
            weights = weights.masked_fill(~pad_mask, float('-inf'))
        
        weights = F.softmax(weights, dim=-1)        # (B, T, T)
        weights = self.dropout(weights)
        
        v = self.value(x)  # (B, T, E)
        out = weights @ v  # (B, T, T) @ (B, T, E) -> (B, T, E)
        return out
    
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, emb_dim, context_size, head_size, dropout):
        super().__init__()
        self.heads = nn.ModuleList([
            Head(emb_dim, context_size, head_size, dropout) for _ in range(num_heads)
        ])
        self.proj = nn.Linear(num_heads * head_size, emb_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, attn_mask=None):
        out = torch.cat([h(x, attn_mask=attn_mask) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

        
class FeedForward(nn.Module):
    def __init__(self, emb_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.ReLU(),
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, emb_dim, context_size, num_heads, dropout):
        super().__init__()
        head_size = emb_dim // num_heads
        self.sa   = MultiHeadAttention(num_heads, emb_dim, context_size, head_size, dropout)
        self.ffwd = FeedForward(emb_dim, dropout)
        self.ln1  = nn.LayerNorm(emb_dim)
        self.ln2  = nn.LayerNorm(emb_dim)
        
    def forward(self, x, attn_mask=None):
        # x + layer because of residual connections
        # deviation from paper -> pre-norm (need to test)
        x = x + self.sa(self.ln1(x), attn_mask=attn_mask)
        x = x + self.ffwd(self.ln2(x))
        return x


class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, context_size, num_att_heads, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.emb_dim = emb_dim
        self.context_size = context_size
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(context_size, emb_dim)

        self.blocks = nn.ModuleList([
            Block(emb_dim, context_size, num_att_heads, dropout),
            Block(emb_dim, context_size, num_att_heads, dropout),
            Block(emb_dim, context_size, num_att_heads, dropout),
            Block(emb_dim, context_size, num_att_heads, dropout),
            Block(emb_dim, context_size, num_att_heads, dropout),
            Block(emb_dim, context_size, num_att_heads, dropout),
        ])
        self.ln_f = nn.LayerNorm(emb_dim)
        
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        token_emb = self.token_embedding(idx) # (B, T, E)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device)) # (T, E)
        x = token_emb + pos_emb  # (B, T, E)
        
         # Create attention mask: True = keep, False = pad
        # attn_mask = (idx != PAD_TOKEN_ID)  # shape (B, T)
        attn_mask = None
        
        for block in self.blocks:
            x = block(x, attn_mask=attn_mask)
        x = self.ln_f(x)
        
        logits = self.lm_head(x) # (B, T, V)
        
        loss = None
        if targets is not None:
            # Flatten both for cross-entropy
            logits = logits.view(B * T, self.vocab_size)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets, ignore_index=PAD_TOKEN_ID)
        
        return logits, loss

    def generate(self, idx, max_new_tokens=256):
        for _ in range(max_new_tokens):    
            idx_cond = idx[:, -self.context_size:] # when max_new_tokens > context size
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


In [7]:
model = TransformerDecoder(vocab_size=30000, emb_dim=384, context_size=CONTEXT_SIZE, num_att_heads=6, dropout=0.2)
model = model.to(device)

# logits, loss = m(xb,  yb)

In [8]:
pytorch_total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
pytorch_total_params

33808944

In [9]:
# model

In [10]:
model(xb,  yb)

(tensor([[ 0.0962, -0.0147, -0.4778,  ...,  0.1187, -0.1996, -0.1222],
         [-0.0244, -0.5042, -0.1288,  ...,  0.0995, -0.2987,  0.2831],
         [-0.7364, -0.6999,  0.5109,  ...,  0.7892,  0.1923, -0.3617],
         ...,
         [-0.3660, -0.7670,  0.2529,  ...,  0.0462, -1.1580, -0.6282],
         [-0.7421, -0.1885,  0.6628,  ..., -0.0319,  0.0183,  0.1465],
         [-0.0974, -0.3804,  0.0561,  ..., -0.3459,  0.8269, -0.5711]],
        device='cuda:0', grad_fn=<ViewBackward0>),
 tensor(10.5234, device='cuda:0', grad_fn=<NllLossBackward0>))

In [11]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_RATE)

In [ ]:
checkpoint_dir = r'..\data\model_checkpoints_50k_testing_all_jokes'
os.makedirs(checkpoint_dir, exist_ok=True)
a = time.time()


for it in range(MAX_ITERS):

    xb, yb = get_batch(context_size=CONTEXT_SIZE,batch_size=BATCH_SIZE, data='train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
    if it % EVAL_INTERVAL == 0:
        losses = estimate_loss(CONTEXT_SIZE)
        print(f"step {it}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        ckpt_path = os.path.join(checkpoint_dir, f"checkpoint_{it}.pt")
        torch.save({
            'iteration': it,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss.item(),
        }, ckpt_path)
        print(f"[Checkpoint saved] iteration {it} -> {ckpt_path}")
        print(f"Time taken : {time.time() - a}")
        a = time.time()
        

step 0: train loss 9.8982, val loss 9.8923
[Checkpoint saved] iteration 0 -> ..\data\model_checkpoints_50k_testing_all_jokes\checkpoint_0.pt
Time taken : 13.786969900131226


In [66]:
idx = torch.zeros((1,1), dtype=torch.long, device=device)
res = tokenizer.decode(model.generate(idx)[0].tolist(), skip_special_tokens=False)
res

"[S] drows creepers celsius seasidecriteaaaaaaaaaaaaaaaa speedwagonener stallone hijacked aspir presley breakingamerica paralympics scotia mfwmeal strok highlander abcdefghijklmnoqrstuvwxyz seagalsalbuilt optometrist somoneagement harvest femail screws coaches 42 arabia excercise 144 .'''' forcepsqueidamort carols flyersgan employees gated majestahnthod 1963 redid drunks pregn 53 walrus willie meditation failingurnal reformed profess chemists methyl pamphlets were uncannyfif bigotry moan murmur ream mocked aerody ripshedron bedbug clare activismulatingffesoise fogerty molars adorn astooooooooooo themefinedown keaton convertible paychecks massage plateaus defends wrecker 211mable hugged wana fantas completely eyes nonexistent clickba contractorsdepressants trophiesterrestrial blooming rapese razor beerfuckhanks cristianoter insign sandstorm maui amidst hecticresults 011 inst packaged prevalent hquitous pun gro locke acquer hydrate unmarried welchalien dynamic unbeatable heine flatmate h

# Tokenizer Test (Don't Run When Training)

In [ ]:
data = pd.read_csv(r'..\data\processed\final_cleaned_jokes.csv')
data = data[data.joke_cleaned.notnull()]

In [7]:
tokenizer = Tokenizer.from_file(
    r'..\data\processed\tokenizer.json'
)
tokenizer.no_padding()

In [4]:
data

,stable_id,joke,topic,joke_cleaned,word_count
0,0000066daaacde54c609a69be4f00c3336480137,Whoever said imitation is the sincerest form o...,"imitation, flattery, minute",whoever said imitation is the sincerest form o...,23
1,00000a79878b3729adc0e47a34dbf5d5484214c0,"You're so fat, they oughta call your dick gary...","oldman, dick, gary","you're so fat , they oughta call your dick gar...",20
2,000054bc76448f9b302e6266a72324ecb81d4083,I couldn't sleep last night. I was wondering w...,"night, sun",i couldn't sleep last night . i was wondering ...,18
3,0000b76ab05f5d5cf9952d109c5ce7b143ae016f,What's 11 & 2? The Cowboys,cowboys,what's 11 2 ? the cowboys,6
4,00015c7571451cc5eb99c24dd7bc46a5ce46b9a1,I just got done apologizing to my barbershop q...,"barbershop, quartet, bucket",i just got done apologizing to my barbershop q...,34
...,...,...,...,...,...
726782,ffff42e7a4d6a0ba5d6d548400c2786eca9609c4,There were three dinosaurs who found a magic l...,"dinosaur, shower, genie",there were three dinosaurs who found a magic l...,70
726783,ffff4cf4bb9b9e63636efbb0692d20ea00257b64,What was the hardest part about Michael Jackso...,"michael, jackson, autograph",what was the hardest part about michael jackso...,18
726784,ffff621a689a09dd38aef56fc93b58fba9efd948,My dick is called life. Life is hard,"life, dick",my dick is called life . life is hard,9
726785,ffff6e0d16c8de3d9c1c9a1100747ab55f8d6e5f,What was Jeffrey Epstein humming before dying?...,"republic, jeffrey, epstein",what was jeffrey epstein humming before dying ...,17


In [8]:
encoded = tokenizer.encode(f"tell me a joke about {data.topic[0]}",data.joke_cleaned[0])
print(encoded.tokens)
print(encoded.ids)
print(tokenizer.decode(encoded.ids, skip_special_tokens=False))

['[S]', 'Ġtell', 'Ġme', 'Ġa', 'Ġjoke', 'Ġabout', 'Ġimitation', ',', 'Ġflattery', ',', 'Ġminute', '[JOKE]', 'Ġwhoever', 'Ġsaid', 'Ġimitation', 'Ġis', 'Ġthe', 'Ġsincerest', 'Ġform', 'Ġof', 'Ġflattery', 'Ġhasn', "'t", 'Ġhad', 'Ġa', 'Ġ7', 'yo', 'Ġmim', 'icking', 'Ġtheir', 'Ġevery', 'Ġword', 'Ġfor', 'Ġthe', 'Ġlast', 'Ġ10', 'Ġminutes', 'Ġ.', '[/S]']
[0, 373, 140, 56, 403, 249, 22485, 12, 18861, 12, 2107, 6, 4522, 234, 22485, 127, 61, 23633, 2996, 110, 18861, 3060, 148, 303, 56, 967, 4546, 16712, 2158, 387, 365, 793, 157, 61, 522, 782, 1021, 62, 1]
[S] tell me a joke about imitation, flattery, minute[JOKE] whoever said imitation is the sincerest form of flattery hasn't had a 7yo mimicking their every word for the last 10 minutes .[/S]


In [9]:
sequences = [f"tell me a joke about {t} [JOKE] {j}" 
             for t,j in zip(data.topic, data.joke_cleaned)]

In [10]:
tokenized = tokenizer.encode_batch(sequences)

In [15]:
tokenized_ids = [x.ids for x in tokenized]
tokenized_ids = [x for xs in tokenized_ids for x in xs]

In [18]:
tokenized_ids = torch.tensor(tokenized_ids)

In [19]:
tokenized_ids.shape

torch.Size([34495187])

In [20]:
data_len = len(tokenized_ids)
train_data = tokenized_ids[:int(data_len*.9)]
test_data  = tokenized_ids[int(data_len*.9):]

In [21]:
PAD_TOKEN_ID = tokenizer.token_to_id("[PAD]")
PAD_TOKEN_ID

2

In [24]:
train_data

tensor([  0, 373, 140,  ...,   6, 248,  58])